# Week 5 · 02 Neural networks: the forward pass

A neural network is logistic regressions stacked in layers, with a nonlinear function between the layers. This notebook starts from a finished network (§E), then shows why one neuron isn't enough (XOR, with a proof), how a hidden layer fixes it, why the activation must be nonlinear, the shapes of a whole layer, and which output layer and loss to use. It ends with your own MLP class, checked number for number against PyTorch. Training (where the weights come from) is `03`. Reading: ISL (the PDF in your repo) §10.1–10.2.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **neuron (unit)**: computes a weighted sum of its inputs plus a bias, $z = w^\top x + b$, then applies an **activation function**, $a = f(z)$.
- **activation function**: the nonlinear function applied to each unit's $z$: sigmoid, tanh, ReLU.
- **layer**: many neurons that read the same inputs. A layer is one weight matrix plus one bias vector.
- **hidden layer**: a layer between the input and the output. Its values aren't in the data; the network computes them.
- **MLP (multilayer perceptron)**: layers stacked so that each feeds the next. Also called a fully connected or dense network.
- **logits**: the last layer's raw scores, before softmax (or sigmoid) turns them into probabilities.
- **forward pass**: computing the output from the input, one layer after another.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "02")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** The path through this notebook. The top row is the idea (why layers), the middle row is the machinery (activations, shapes, output and loss), the bottom row is practice.

In [ ]:
fig_nn_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and the logistic gradient from memory | a neuron **is** logistic regression, so start from it |
| E | the finished network on a new input | start from the end: what a network stores and how it predicts, before how it's built |
| 1 | one neuron = logistic regression | the building block, in network words |
| 2 | XOR: one neuron can't | a proof of why we need more than one layer |
| 3 | a hidden layer bends the space | how two layers solve what one can't |
| 4 | activations: why nonlinear | without them, any number of layers collapses into one (a proof) |
| 5 | layers as matrices | the shapes of a whole batch, layer by layer |
| 6 | output layers and losses | they come from maximum likelihood, not from taste |
| 7 | softmax + cross-entropy: the gradient $p - y$ | the gradient every classifier's backprop starts from (`03`) |
| 8 | counting parameters | a standard exam question |
| 9 | universal approximation | what one hidden layer can represent, in a picture |
| 10 | your MLP class vs PyTorch | from scratch next to the package, same numbers |

## 0. Recap first · 10 min, closed book

**Quick check:** Logistic regression: the model for $P(y = 1 \mid x)$, and the loss for one sample.

<details><summary>Answer</summary>

$p = \sigma(w^\top x + b)$ with $\sigma(z) = \frac{1}{1 + e^{-z}}$. Loss: $-[y \log p + (1 - y)\log(1 - p)]$ (binary cross-entropy). [Week 3 · 01 §5 ↗](../week3/01_logistic_regression_in_pictures.ipynb)

</details>

**Quick check:** Its gradient with respect to $w$, averaged over $n$ samples?

<details><summary>Answer</summary>

$\frac{1}{n} X^\top (p - y)$, and $\frac{1}{n}\sum_i (p_i - y_i)$ for $b$. Each point pulls with strength $p_i - y_i$. [Week 3 · 01 §6 ↗](../week3/01_logistic_regression_in_pictures.ipynb)

</details>

**Quick check:** $\sigma'(z) = $ ?

<details><summary>Answer</summary>

$\sigma(z)\,(1 - \sigma(z))$. Its largest value is $0.25$, at $z = 0$.

</details>

**Quick check:** Maximum likelihood: what is minimized, in words?

<details><summary>Answer</summary>

The negative log of the probability the model gives to the observed data. For independent samples it's a sum over samples. [Week 1 · 03 §4 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** Softmax of $z = (0, \ln 2)$?

<details><summary>Answer</summary>

$\left(\frac{1}{1 + 2}, \frac{2}{1 + 2}\right) = (0.333, 0.667)$, since $e^0 = 1$ and $e^{\ln 2} = 2$.

</details>

**Exercise 1 · The logistic regression gradient, from memory.** $p = \sigma(Xw + b)$; return `(dw, db)` with $dw = \frac{1}{n} X^\top (p - y)$, **shape `(d,)`**, and $db = \frac{1}{n}\sum_i (p_i - y_i)$, a number. `X` `(n, d)`, `y` `(n,)` of 0/1, `w` `(d,)`, `b` a float. (The mini-project's from-scratch core, which you skipped.)

**Example**: what goes in, and what should come out:
```python
X = np.array([[1.0, 2.0], [2.0, 0.0], [0.0, 1.0]]); y = np.array([1.0, 0.0, 1.0])
dw, db = logistic_grad(X, y, np.array([0.5, -0.5]), 0.0)
dw    # → [ 0.28  -0.622]
db    # → -0.171
```

In [ ]:
def logistic_grad(X, y, w, b):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: the finished network, and a new input

Below is a finished network for XOR (output 1 when exactly one of two inputs is 1). Its numbers were set by hand so that you can read them; training (`03`) finds numbers that do the same job. **What it stores:**

| Stored | Shape | Value | What it is | Built in |
|---|---|---|---|---|
| $W_1$ | $(2, 2)$ | $\begin{pmatrix} 6 & -6 \\ 6 & -6 \end{pmatrix}$ | column $k$ = hidden unit $k$'s weights on $(x_1, x_2)$ | §3 |
| $b_1$ | $(2,)$ | $(-3,\ 9)$ | one bias per hidden unit | §3 |
| $W_2$ | $(2, 1)$ | $\begin{pmatrix} 8 \\ 8 \end{pmatrix}$ | the output unit's weights on $(h_1, h_2)$ | §3 |
| $b_2$ | $(1,)$ | $-12$ | the output unit's bias | §3 |

**Predicting a new input** $x = (0.9, 0.2)$: one layer at a time, a weighted sum, then the sigmoid.

- Hidden sums: $z_1 = x W_1 + b_1 = (6 \cdot 0.9 + 6 \cdot 0.2 - 3,\ -6 \cdot 0.9 - 6 \cdot 0.2 + 9) = (3.6,\ 2.4)$.
- Hidden values: $h = \sigma(z_1) = (0.973,\ 0.917)$.
- Output sum: $z_2 = h W_2 + b_2 = 8 \cdot 0.973 + 8 \cdot 0.917 - 12 = 3.122$.
- Output: $p = \sigma(3.122) = 0.958$, so class 1 (0.9 is "on", 0.2 is "off": exactly one input is on).

🖼 **Picture.** The same computation drawn as a network: weights on the edges, biases next to the units, each unit's value inside it.

In [ ]:
fig_xor_network()

In [ ]:
W1 = np.array([[6.0, -6.0], [6.0, -6.0]])   # (2 inputs, 2 hidden units): column k = unit k's weights
b1 = np.array([-3.0, 9.0])                  # (2,): one bias per hidden unit
W2 = np.array([[8.0], [8.0]])               # (2 hidden, 1 output)
b2 = np.array([-12.0])                      # (1,)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

x = np.array([[0.9, 0.2]])        # (1, 2): ONE new sample, kept 2-D (a batch of size 1)
z1 = x @ W1 + b1                  # (1, 2)
h = sigmoid(z1)                   # (1, 2)
z2 = h @ W2 + b2                  # (1, 1)
p = sigmoid(z2)                   # (1, 1)
print(z1, h.round(3), z2.round(3), p.round(3))   # → [[3.6 2.4]] [[0.973 0.917]] [[3.122]] [[0.958]]

The same network in PyTorch (`06` explains every line). `nn.Linear(2, 2)` is one layer; it stores its weight as **(out, in)**, the transpose of our $W_1$, so we copy `W1.T` into it:

In [ ]:
import torch
import torch.nn as nn
net = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 1), nn.Sigmoid())   # applied in this order
with torch.no_grad():                               # we're setting numbers, not training
    net[0].weight.copy_(torch.tensor(W1.T))         # (out, in) = (2, 2)
    net[0].bias.copy_(torch.tensor(b1))
    net[2].weight.copy_(torch.tensor(W2.T))         # (1, 2)
    net[2].bias.copy_(torch.tensor(b2))
print(round(net(torch.tensor([[0.9, 0.2]])).item(), 3))   # → 0.958

**The rest of this notebook answers:** why does a hidden layer make XOR possible (§1–3), why is there a sigmoid between the layers (§4), what changes for many samples and many classes (§5–7), and how large such networks get (§8–9). Where $W_1, b_1, W_2, b_2$ come from, i.e. training, is `03`.

## 1. One neuron = logistic regression

> ↩ **Recap** · [Week 3 · 01 §2–3 (score, then sigmoid) ↗](../week3/01_logistic_regression_in_pictures.ipynb)
>
> Logistic regression computes a score $z = w^\top x + b$ (a height over the plane), then turns it into a probability $p = \sigma(z)$. The boundary $p = 0.5$ is the line $z = 0$.

A neuron is exactly this computation, with network names for the parts:

| Logistic regression | Neural network |
|---|---|
| coefficients $w$ | weights |
| intercept $b$ | bias |
| score $z = w^\top x + b$ | pre-activation |
| $\sigma$ | activation function |
| $p = \sigma(z)$ | the neuron's output (activation) |

A **layer** of $h$ neurons reading the same $d$ inputs is $h$ logistic regressions side by side: their weight vectors are the $h$ columns of one matrix $W$, shape $(d, h)$. Check that a neuron really is logistic regression: put scikit-learn's fitted coefficients into the neuron formula.

In [ ]:
from sklearn.linear_model import LogisticRegression
Xd = np.array([[1, 1], [2, 1], [1, 3], [3, 3], [4, 2], [4, 4.0]]); yd = np.array([0, 0, 0, 1, 1, 1])
clf = LogisticRegression().fit(Xd, yd)
neuron = sigmoid(Xd @ clf.coef_.ravel() + clf.intercept_)    # coef_ is (1, d): ravel() makes it (d,)
print(np.allclose(neuron, clf.predict_proba(Xd)[:, 1]))     # → True

## 2. XOR: one neuron can't

🖼 **Picture.** The four XOR points. The dashed, dotted and dash-dot lines are three attempts; each one leaves a blue and an orange point on the same side.

In [ ]:
fig_xor_problem()

**Proposition.** No single neuron classifies all four XOR points correctly: there are no $w_1, w_2, b$ with $z = w_1 x_1 + w_2 x_2 + b \ge 0$ exactly for $(0, 1)$ and $(1, 0)$.

**Proof.** Suppose such $w_1, w_2, b$ exist. The four points require:

$$
\begin{aligned} (0, 0) \to 0: &\quad b < 0, \\ (1, 1) \to 0: &\quad w_1 + w_2 + b < 0, \\ (0, 1) \to 1: &\quad w_2 + b \ge 0, \\ (1, 0) \to 1: &\quad w_1 + b \ge 0. \end{aligned}
$$

Adding the last two lines gives $w_1 + w_2 + 2b \ge 0$. Adding the first two gives $w_1 + w_2 + 2b < 0$. Both can't hold. $\blacksquare$

**Remarks.** (1) The sigmoid doesn't help: it's increasing, so $\sigma(z) \ge 0.5$ exactly when $z \ge 0$, and the boundary is still the line $z = 0$. (2) The proof works for **every** linear classifier: logistic regression, a linear SVM, the perceptron. XOR is the classic example of data that isn't linearly separable. (3) Adding the feature $x_1 x_2$ by hand would fix it (a feature map, as with kernels in Week 3). A hidden layer **learns** such features instead of you choosing them; that's the next section.

## 3. A hidden layer bends the space

**The idea.** XOR is "at least one is on" **and** "not both are on": $\text{XOR} = \text{OR} \wedge \text{NAND}$. Each of OR, NAND and AND **is** linearly separable, so each can be one neuron. Two hidden neurons compute OR and NAND; the output neuron computes AND of their outputs.

| $x_1$ | $x_2$ | OR | NAND | AND(OR, NAND) = XOR |
|---|---|---|---|---|
| 0 | 0 | 0 | 1 | 0 |
| 0 | 1 | 1 | 1 | 1 |
| 1 | 0 | 1 | 1 | 1 |
| 1 | 1 | 1 | 0 | 0 |

That's what §E's weights do: $h_1 = \sigma(6x_1 + 6x_2 - 3)$ is close to 1 once either input is on (OR); $h_2 = \sigma(-6x_1 - 6x_2 + 9)$ is close to 1 unless both are on (NAND); the output $\sigma(8h_1 + 8h_2 - 12)$ needs both (AND). The four inputs, through the network:

| Input | $z_1 = xW_1 + b_1$ | $h = \sigma(z_1)$ | $z_2 = hW_2 + b_2$ | $p$ | XOR |
|---|---|---|---|---|---|
| (0, 0) | (−3, 9) | (0.047, 1.000) | −3.622 | 0.026 | 0 |
| (0, 1) | (3, 3) | (0.953, 0.953) | 3.241 | 0.962 | 1 |
| (1, 0) | (3, 3) | (0.953, 0.953) | 3.241 | 0.962 | 1 |
| (1, 1) | (9, −3) | (1.000, 0.047) | −3.622 | 0.026 | 0 |

🖼 **Picture.** Left: in the input space the network's "class 1" region (orange) is the band between the two hidden units' lines, which no single line could make. Right: the hidden layer moves each point to new coordinates $(h_1, h_2)$; there the classes **are** separated by one line, and the output neuron is just logistic regression on $(h_1, h_2)$. The star is §E's new input.

In [ ]:
fig_xor_hidden_space()

**This is the whole idea of deep learning:** the hidden layers learn a new representation of the input in which a simple (linear) model works. With more layers, each one builds on the previous one's features.

**Training finds such weights by itself.** scikit-learn's `MLPClassifier` with one hidden layer of 2 sigmoid units, trained on the four points:

In [ ]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(2,),   # one hidden layer with 2 units
                    activation="logistic",      # the sigmoid between the layers
                    solver="lbfgs",             # an optimizer for tiny problems (05 covers SGD and Adam)
                    max_iter=2000, random_state=0).fit(XOR_X, XOR_Y)   # XOR_X (4, 2), XOR_Y (4,) come from figures.py
print(mlp.predict(XOR_X))                      # → [0 1 1 0]

Its learned weights (`mlp.coefs_[0]`) are different numbers from ours, but they do the same job: two lines that cut out the band. There are many solutions; training lands on one of them.

## 4. Activations: why they must be nonlinear

🖼 **Picture.** The three common activations (blue) and their derivatives (orange, dashed). The derivative matters for training: backpropagation multiplies by it at every layer (`03`).

In [ ]:
fig_activations()

| Name | Formula | Derivative | Range | Used for | Watch out for |
|---|---|---|---|---|---|
| sigmoid | $\sigma(z) = \frac{1}{1 + e^{-z}}$ | $\sigma(z)(1 - \sigma(z))$ | (0, 1) | the output of a binary classifier | derivative $\le 0.25$ and $\to 0$ for large $\lvert z \rvert$: gradients vanish through many layers |
| tanh | $\frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $1 - \tanh^2(z)$ | (−1, 1) | hidden layers of recurrent networks (Week 8) | also flattens for large $\lvert z \rvert$ |
| ReLU | $\max(0, z)$ | 1 if $z > 0$, else 0 | $[0, \infty)$ | the default for hidden layers | a unit with $z < 0$ for every input gets no gradient (a "dead" unit) |

(Two variants you may see in code: Leaky ReLU, $\max(0.01z, z)$, which keeps a small slope for $z < 0$; and GELU, a smooth ReLU used in Transformers.)

**Why must there be an activation at all?** Because without one, stacking layers gains nothing:

**Proposition.** Two linear layers in a row are one linear layer: for any $W_1$ $(d, h)$, $b_1$ $(h,)$, $W_2$ $(h, m)$, $b_2$ $(m,)$ there are $W = W_1 W_2$ $(d, m)$ and $b = b_1 W_2 + b_2$ $(m,)$ with

$$
(x W_1 + b_1) W_2 + b_2 = x W + b \quad \text{for every } x.
$$

**Proof.**

$$
\begin{aligned} (x W_1 + b_1) W_2 + b_2 &= x W_1 W_2 + b_1 W_2 + b_2 && \text{(matrix multiplication distributes)} \\ &= x\, \underbrace{(W_1 W_2)}_{W\ (d,\ m)} + \underbrace{(b_1 W_2 + b_2)}_{b\ (m,)}. && \blacksquare \end{aligned}
$$

**Remarks.** (1) By induction, **any number** of linear layers is one linear layer. A deep network without activations is just logistic regression, and §2's proof says it can't do XOR. (2) The nonlinearity doesn't need to be complicated: ReLU is linear on each side of 0, and that bend is enough (§9). (3) In math books and in PyTorch's storage, layers are written $W x$ with $W$ of shape (out, in); here, rows are samples, so it's $x W$ with $W$ (in, out). Same layer, transposed.

**Example, checked two ways.** $W_1 = \begin{pmatrix} 1 & -1 & 2 \\ 0 & 3 & 1 \end{pmatrix}$, $b_1 = (1, 0, -1)$, $W_2 = (2, 1, -1)^\top$, $b_2 = 0.5$, $x = (1, 2)$:

In [ ]:
W1_ = np.array([[1.0, -1.0, 2.0], [0.0, 3.0, 1.0]]); b1_ = np.array([1.0, 0.0, -1.0])
W2_ = np.array([[2.0], [1.0], [-1.0]]); b2_ = np.array([0.5])
x_ = np.array([[1.0, 2.0]])
print((x_ @ W1_ + b1_) @ W2_ + b2_, x_ @ (W1_ @ W2_) + (b1_ @ W2_ + b2_))   # → [[6.5]] [[6.5]]

🖼 **Picture.** The same network shape (two hidden layers of 16 units) trained on the moons. Without activations the boundary is a straight line, as the proposition says; with ReLU it bends around the data.

In [ ]:
fig_linear_vs_relu()

**Exercise 2 · Collapse two linear layers.** Return `(W, b)` with $W = W_1 W_2$, **shape `(d, m)`**, and $b = b_1 W_2 + b_2$, **shape `(m,)`**, so that `X @ W + b` equals `(X @ W1 + b1) @ W2 + b2` for every `X`. `W1` `(d, h)`, `b1` `(h,)`, `W2` `(h, m)`, `b2` `(m,)`.

**Example**: what goes in, and what should come out:
```python
W, b = collapse_linear(W1_, b1_, W2_, b2_)    # the numbers from the cell above
W    # → [[-1.]
     #    [ 2.]]
b    # → [3.5]
```

In [ ]:
def collapse_linear(W1, b1, W2, b2):
    pass  # your code

In [ ]:
check(2)

## 5. Layers as matrices: the shapes of a batch

**The question:** what does a two-layer network compute for a whole batch of $n$ samples at once, and what shape is everything? With $d$ inputs, $h$ hidden units and $C$ classes:

$$
Z_1 = X W_1 + b_1, \quad H = \text{ReLU}(Z_1), \quad Z_2 = H W_2 + b_2, \quad P = \operatorname{softmax}(Z_2)\ \text{(each row)}
$$

| Symbol | Shape | A row is | A column is | An entry is |
|---|---|---|---|---|
| $X$ | $(n, d)$ | one sample | one input feature | $x_{ij}$ |
| $W_1$ | $(d, h)$ | one input feature | one hidden unit | weight from input $j$ to hidden unit $k$ |
| $b_1$ | $(h,)$ |  |  | hidden unit $k$'s bias |
| $Z_1, H$ | $(n, h)$ | one sample | one hidden unit | unit $k$'s sum / value for sample $i$ |
| $W_2$ | $(h, C)$ | one hidden unit | one class | weight from hidden unit $k$ to class $c$ |
| $b_2$ | $(C,)$ |  |  | class $c$'s bias |
| $Z_2$ | $(n, C)$ | one sample | one class | logit $z_{ic}$ |
| $P$ | $(n, C)$ | one sample | one class | $P(\text{class } c \mid x_i)$ |

🖼 **Picture.** A batch of 4 samples through a network with 2 inputs, 3 hidden units and 2 classes. The rows never mix: sample $i$ stays row $i$.

In [ ]:
fig_mlp_shapes()

Each layer is `01` §7's dense layer, `H @ W + b`, followed by an activation. The whole XOR table of §3 in one go, by feeding all four inputs as one batch:

In [ ]:
Z1 = XOR_X @ W1 + b1         # (4, 2) @ (2, 2) + (2,) -> (4, 2)
H = sigmoid(Z1)                # (4, 2)
P = sigmoid(H @ W2 + b2)       # (4, 2) @ (2, 1) + (1,) -> (4, 1)
print(P.ravel().round(3))      # → [0.026 0.962 0.962 0.026]

**Recipe (the forward pass of an MLP).** `H = X`; for every layer: `Z = H @ W + b`, then `H = activation(Z)`, except after the **last** layer, whose output is the logits. Softmax (or sigmoid) is applied by the loss or at prediction time.

**Exercise 3 · MLP forward pass.** `Ws` and `bs` are lists of the layers' weights `(size_in, size_out)` and biases `(size_out,)`. Apply `H @ W + b` layer by layer, with ReLU, $\max(0, z)$, after every layer **except the last**. Return the logits, **shape `(n, size of the last layer)`**.

**Example**: what goes in, and what should come out:
```python
X = np.array([[1.0, 2.0], [1.0, -1.0]])         # (2, 2): two samples
mlp_forward(X, [W1_, W2_], [b1_, b2_])          # W1_ ... from §4
# → [[6.5]
#    [4.5]]       (without ReLU the second row would be 0.5)
```

In [ ]:
def mlp_forward(X, Ws, bs):
    pass  # your code

In [ ]:
check(3)

## 6. Output layers and losses come from maximum likelihood

> ↩ **Recap** · [Week 1 · 03 §4 (maximum likelihood) ↗](../week1/03_math_essentials.ipynb)
>
> Choose the parameters that make the observed data most probable: maximize $\prod_i P(y_i \mid x_i)$, i.e. minimize the negative log-likelihood $-\sum_i \log P(y_i \mid x_i)$.

**The question:** which output layer and which loss? The answer is not a matter of taste: pick a probability model for $y$ given the network's output, and the loss is its negative log-likelihood.

| Task | Output layer | Model for $y$ | Loss = − log-likelihood | PyTorch |
|---|---|---|---|---|
| regression | 1 unit, no activation | $y \sim \mathcal{N}(\hat y, \sigma^2)$ | $(y - \hat y)^2$ (mean squared error, up to constants) | `nn.MSELoss` |
| two classes | 1 unit + sigmoid | $y \sim \text{Bernoulli}(p)$ | $-[y \log p + (1 - y)\log(1 - p)]$ | `nn.BCEWithLogitsLoss` |
| $C$ classes | $C$ units + softmax | $y \sim \text{Categorical}(p_1, \dots, p_C)$ | $-\log p_y$ (cross-entropy) | `nn.CrossEntropyLoss` |

**Proposition.** If $y_i \in \{0, 1\}$ are independent with $P(y_i \mid x_i) = p_i^{y_i}(1 - p_i)^{1 - y_i}$, the negative log-likelihood is the sum of the binary cross-entropies.

**Proof.**

$$
-\log \prod_{i=1}^{n} p_i^{y_i}(1 - p_i)^{1 - y_i} = -\sum_{i=1}^{n} \log\left(p_i^{y_i}(1 - p_i)^{1 - y_i}\right) = -\sum_{i=1}^{n} \Big[y_i \log p_i + (1 - y_i)\log(1 - p_i)\Big]. \qquad \blacksquare
$$

**Remarks.** (1) The formula $p^{y}(1 - p)^{1 - y}$ is just a compact way to write "$p$ if $y = 1$, $1 - p$ if $y = 0$". (2) The Gaussian model gives the squared error the same way: $-\log \mathcal{N}(y \mid \hat y, \sigma^2) = \frac{(y - \hat y)^2}{2\sigma^2} + \text{const}$. (3) Dividing by $n$ (the mean instead of the sum) doesn't change the minimizer; PyTorch's losses use the mean by default. See [Week 2 · 02 §6 ↗](../week2/02_map_and_entropy_in_depth.ipynb) for cross-entropy from the information side.

## 7. Softmax + cross-entropy: the gradient is $p - y$

**The question:** training needs $\partial L / \partial z$, the gradient of the loss with respect to the logits; it's where backpropagation starts (`03`). For softmax with cross-entropy it is remarkably simple.

**Proposition.** Let $z \in \mathbb{R}^C$ be the logits, $p = \operatorname{softmax}(z)$, $y$ the true class, and $L = -\log p_y$. Then for every class $j$:

$$
\frac{\partial L}{\partial z_j} = p_j - \mathbb{1}[j = y], \qquad\text{i.e.}\qquad \nabla_z L = p - e_y,
$$

where $e_y$ is the one-hot vector of the true class.

**Proof.** Write the loss without the fraction: since $p_y = e^{z_y} / \sum_c e^{z_c}$,

$$
\begin{aligned} L &= -\log p_y = -z_y + \log \sum_{c} e^{z_c}, \\ \frac{\partial L}{\partial z_j} &= -\frac{\partial z_y}{\partial z_j} + \frac{1}{\sum_c e^{z_c}} \cdot \frac{\partial}{\partial z_j}\sum_c e^{z_c} = -\mathbb{1}[j = y] + \frac{e^{z_j}}{\sum_c e^{z_c}} = p_j - \mathbb{1}[j = y]. \qquad \blacksquare \end{aligned}
$$

**Remarks.** (1) Every component lies between $-1$ and $1$, so this gradient never explodes. (2) With two classes and one logit it's logistic regression's $p - y$ again. (3) For a batch with the **mean** loss, the gradient with respect to $Z$ $(n, C)$ is $G = \frac{1}{n}(P - Y)$, with $Y$ the one-hot labels: this $G$ is what `03` passes backwards through the layers. (4) It's why `nn.CrossEntropyLoss` takes the **logits**: it does log-softmax and cross-entropy together, which is numerically stable and has this simple gradient. Applying softmax yourself first is a classic bug (`06` §4).

**Example, checked three ways.** $z = (2, 1, 0.1)$, true class $y = 0$:

| $c$ | $z_c$ | $e^{z_c}$ | $p_c = e^{z_c} / 11.212$ | $p_c - \mathbb{1}[c = y]$ |
|---|---|---|---|---|
| 0 (true) | 2 | 7.389 | 0.659 | −0.341 |
| 1 | 1 | 2.718 | 0.242 | 0.242 |
| 2 | 0.1 | 1.105 | 0.099 | 0.099 |

So $L = -\log 0.659 = 0.417$. The formula, a numerical derivative, and PyTorch's autograd all agree:

In [ ]:
z = np.array([2.0, 1.0, 0.1]); y = 0
p = np.exp(z) / np.exp(z).sum()
grad_formula = p - np.eye(3)[y]
loss = lambda z_: -np.log(np.exp(z_[y]) / np.exp(z_).sum())
grad_numeric = np.array([(loss(z + e) - loss(z - e)) / 2e-6 for e in np.eye(3) * 1e-6])   # nudge one logit at a time
zt = torch.tensor(z, requires_grad=True)
torch.nn.functional.cross_entropy(zt[None], torch.tensor([y])).backward()   # zt[None]: a batch of 1, shape (1, 3)
print(grad_formula.round(3), grad_numeric.round(3), zt.grad.numpy().round(3))
# → [-0.341  0.242  0.099] [-0.341  0.242  0.099] [-0.341  0.242  0.099]

🖼 **Picture.** Logits → probabilities → gradient. Gradient descent moves $z$ **against** the gradient: the true class's logit goes up by the most, the others go down in proportion to how much probability they took.

In [ ]:
fig_softmax_grad()

**Exercise 4 · The softmax-cross-entropy gradient for a batch.** $G = \frac{1}{n}(P - Y)$: `P` = softmax of each row of the logits `Z` `(n, C)`, `Y` = one-hot of the integer labels `y` `(n,)`. Return **shape `(n, C)`**. This is $\partial L / \partial Z$ for the **mean** cross-entropy over the batch.

**Example**: what goes in, and what should come out:
```python
softmax_ce_grad(np.array([[2.0, 1.0, 0.1]]), np.array([0]))    # → [[-0.341  0.242  0.099]]   n = 1
softmax_ce_grad(np.array([[2.0, 1.0, 0.1], [0.0, 0.0, 0.0]]), np.array([0, 2]))
# → [[-0.17   0.121  0.049]
#    [ 0.167  0.167 -0.333]]   each row (p - y) divided by n = 2
```

In [ ]:
def softmax_ce_grad(Z, y):
    pass  # your code

In [ ]:
check(4)

## 8. Counting parameters

A layer from $a$ units to $b$ units has an $(a, b)$ weight matrix and $b$ biases: $a \cdot b + b$ parameters. For an MNIST classifier with layer sizes $784 \to 128 \to 64 \to 10$:

| Layer | Weights | Biases | Total |
|---|---|---|---|
| 784 → 128 | 784 × 128 = 100,352 | 128 | 100,480 |
| 128 → 64 | 128 × 64 = 8,192 | 64 | 8,256 |
| 64 → 10 | 64 × 10 = 640 | 10 | 650 |
| all |  |  | **109,386** |

In [ ]:
net = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
print(sum(p.numel() for p in net.parameters()))     # → 109386   numel() = number of entries of a tensor

> ⚠️ **Common mistake.** Forgetting the biases (the answer would be 109,184), or counting the activation layers (ReLU has no parameters).

**Exercise 5 · Count the parameters.** `sizes` lists the layer sizes, input first, e.g. `[784, 128, 64, 10]`. Return the number of weights plus biases, an `int`.

**Example**: what goes in, and what should come out:
```python
count_params([784, 128, 64, 10])    # → 109386
count_params([2, 2, 1])             # → 9   (the XOR network: 4 + 2 + 2 + 1)
```

In [ ]:
def count_params(sizes):
    pass  # your code

In [ ]:
check(5)

## 9. What one hidden layer can represent

**Universal approximation theorem** (Cybenko 1989, Hornik 1991; informal). A network with **one** hidden layer and a non-polynomial activation can approximate any continuous function on a bounded region as closely as you like, if it has enough hidden units.

🖼 **Picture.** Why, for ReLU: each hidden unit $\max(0, x - t)$ adds one bend at $t$, so $K$ units give a piecewise-linear curve with $K$ bends (here the output weights were fitted by least squares). More units, more bends, a closer fit.

In [ ]:
fig_universal()

**Remarks.** (1) The theorem says such weights **exist**; it doesn't say training will find them, or how many units you need (it can be astronomically many). (2) Deep networks can represent some functions with exponentially fewer units than a single wide layer, which is one reason depth helps. (3) Fitting the training data is not the goal; generalizing is. Regularization (Week 6) is about that gap.

## 10. Your MLP class, next to PyTorch

**The design.** The class stores the section-E numbers: a list `W` of weight matrices and a list `b` of bias vectors, one per layer. `forward` returns the logits; `predict_proba` applies softmax; `predict` takes the most probable class.

| Attribute / method | What it holds / returns | Shape |
|---|---|---|
| `self.W[l]` | layer $l$'s weights | `(sizes[l], sizes[l+1])` |
| `self.b[l]` | layer $l$'s biases, starting at 0 | `(sizes[l+1],)` |
| `forward(X)` | the logits (ReLU between layers, none after the last) | `(n, sizes[-1])` |
| `predict_proba(X)` | softmax of each row of the logits | `(n, sizes[-1])` |
| `predict(X)` | the index of the largest logit in each row | `(n,)` |

**Why random starting weights?** If every weight started equal, every hidden unit would compute the same thing, get the same gradient, and stay identical forever: $h$ units would act like one. Random numbers break the symmetry. **Why scale them by $\sqrt{2 / \text{fan-in}}$** (He initialization; fan-in = the number of inputs to the layer)? So that each layer's outputs have roughly the same spread as its inputs: too large and the values grow layer after layer, too small and they shrink towards 0 (the start of `03` §7's vanishing gradients). Use `rng = np.random.default_rng(seed)` and `rng.normal(size=(a, b)) * np.sqrt(2 / a)`.

**Exercise 6 · Your MLP class.** `MyMLP(sizes, seed=0)`: `self.W` = list of He-initialized weights `(sizes[l], sizes[l+1])`, `self.b` = list of zero biases `(sizes[l+1],)`. Methods `forward(X)` (logits; ReLU between layers), `predict_proba(X)` (softmax per row) and `predict(X)` (argmax per row). Reuse your exercise-3 logic.

**Example**: what goes in, and what should come out:
```python
m = MyMLP([2, 3, 2], seed=0)
[w.shape for w in m.W]                  # → [(2, 3), (3, 2)]
m.predict_proba(np.array([[1.0, 2.0], [0.5, -1.0]])).sum(axis=1)    # → [1. 1.]   each row sums to 1
```

In [ ]:
class MyMLP:
    def __init__(self, sizes, seed=0):
        pass  # your code

    def forward(self, X):
        pass  # your code

    def predict_proba(self, X):
        pass  # your code

    def predict(self, X):
        pass  # your code

In [ ]:
check(6)

**Same numbers in PyTorch** (run this after exercise 6 passes). Build the same layers with `nn.Linear`, copy your weights in (transposed: PyTorch stores (out, in)), and compare the logits:

In [ ]:
mine = MyMLP([4, 8, 3], seed=1)
torch_net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
with torch.no_grad():
    for layer, W, b in zip([torch_net[0], torch_net[2]], mine.W, mine.b):
        layer.weight.copy_(torch.tensor(W.T))      # our (in, out) -> PyTorch's (out, in)
        layer.bias.copy_(torch.tensor(b))
Xt = np.random.default_rng(2).normal(size=(5, 4))
out_torch = torch_net(torch.tensor(Xt, dtype=torch.float32)).detach().numpy()   # detach(): no gradient needed, then to NumPy
print(np.allclose(mine.forward(Xt), out_torch, atol=1e-5))    # → True

**Recipe (an MLP from scratch).** Store lists of weights `(in, out)` and zero biases; He-initialize the weights; forward = dense layer + ReLU per layer, no activation at the end; softmax for probabilities; argmax for the class. To compare with PyTorch, transpose each weight.

In [ ]:
for n in range(1, 7):
    check(n)

---
Solutions: `solutions/02_solutions.py`. Next: `03_backprop.ipynb`, where these weights get trained.